# Case Study — Module M5: Machine Learning Fundamentals
## Data Preprocessing and Feature Scaling: RetailData Analytics

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phases** | Phase 1–3: Business Understanding + Data Understanding + Data Preparation |
| **Module** | M5 — Machine Learning Fundamentals (Alkemy Bootcamp) |
| **Dataset** | RetailData Analytics — Customer Demographics Fragment (4 rows, synthetic) |
| **Case** | L3 — Preprocessing and Feature Scaling |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> RetailData Analytics has received a customer demographics dataset with missing values, unencoded categorical variables, and inconsistent numeric scales — all of which would degrade predictive model performance. Applying CRISP-DM + LEAN, this case delivers a complete preprocessing pipeline: mean imputation for missing income, Label Encoding and One-Hot Encoding for city names, and both Min-Max normalization and Z-Score standardization for numeric features. The result is a model-ready dataset and a documented rationale for each transformation decision.

## Table of Contents

1. [Phase 1 — Business Understanding](#phase-1)
2. [Phase 2 — Data Understanding](#phase-2)
3. [Phase 3 — Data Preparation](#phase-3)
   - [3.1 Missing Value Imputation](#imputation)
   - [3.2 Categorical Encoding: Label Encoding](#label-encoding)
   - [3.3 Categorical Encoding: One-Hot Encoding](#ohe)
   - [3.4 Dummy Variables](#dummy)
   - [3.5 Feature Scaling: Min-Max Normalization](#minmax)
   - [3.6 Feature Scaling: Z-Score Standardization](#zscore)
   - [3.7 Scaling Comparison Visualization](#visualization)
4. [Phase 5 — Evaluation & Reflection](#phase-5)
5. [LEAN Filter — Waste Elimination Review](#lean-filter)
6. [Decisions Log](#decisions-log)
7. [Next Steps](#next-steps)

---

## 1. Phase 1 — Business Understanding <a id='phase-1'></a>

### Problem Statement Canvas

| Element | Description |
|---|---|
| **Business Problem** | A supermarket chain wants to predict which customers are most likely to make recurring purchases. Raw data is not model-ready: missing values, unencoded categories, and inconsistent scales. |
| **Business Impact** | Without preprocessing, any ML model trained on this data will produce biased predictions, lower accuracy, and unreliable customer segmentation — leading to poor marketing ROI. |
| **Decision to Support** | Which customers should receive loyalty campaigns or targeted promotions to maximize repeat purchase rate? |
| **Analytical Question** | Can we transform the raw customer dataset into a clean, encoded, and scaled feature matrix suitable for a supervised classification model? |
| **Success Metrics** | Zero missing values after imputation. All categorical variables numerically encoded. Numeric features scaled to [0,1] (Min-Max) and standardized (Z-Score). |
| **Proposed Approach** | Apply mean imputation → Label Encoding → One-Hot Encoding → Dummy variables → Min-Max normalization → Z-Score standardization using scikit-learn and pandas. |

### Personal Perspective — ICI Background

From an operations engineering standpoint, data preprocessing is equivalent to **incoming quality control** in a manufacturing process: raw inputs (data) must pass inspection and transformation before entering the production line (the model). Skipping this step is the data equivalent of running defective materials through a CNC machine — the output will be unreliable regardless of the machine's sophistication. LEAN principle: **eliminate defects at the source**, not downstream.

---

## 2. Phase 2 — Data Understanding <a id='phase-2'></a>

In [ ]:
# =============================================================================
# IMPORTS
# Standard Library
import warnings

# Core Data Science
import numpy as np
import pandas as pd

# ML — Preprocessing
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

print('Environment ready.')

In [ ]:
# =============================================================================
# RAW DATASET — as provided by the client
# Source: RetailData Analytics case — L3 fragment
# =============================================================================

data_raw = {
    'ID':       [1,        2,         3,       4        ],
    'Age':      [25,       45,        30,      40       ],
    'City':     ['Madrid', 'Sevilla', 'Madrid', 'Barcelona'],
    'Income':   [30000,    50000,     np.nan,  40000    ]
}

df = pd.DataFrame(data_raw)

print('=== Raw Dataset ===')
print(df)
print()
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# =============================================================================
# DATA QUALITY AUDIT — CRISP-DM Phase 2
# =============================================================================

print('=== Data Quality Audit ===')
print(f'Total rows:          {len(df)}')
print(f'Total columns:       {len(df.columns)}')
print(f'Missing values:      {df.isnull().sum().sum()} '
      f'({df.isnull().sum().sum() / df.size * 100:.1f}% of all cells)')
print(f'Duplicate rows:      {df.duplicated().sum()}')
print()
print('Numeric summary:')
print(df[['Age', 'Income']].describe())
print()
print('Unique cities:', df['City'].unique())

### Data Quality Findings

| Issue | Column | Detail | Action |
|---|---|---|---|
| Missing value | `Income` | Row 3 (ID=3, Madrid) is NaN — 25% of income values | Mean imputation |
| Unencoded categorical | `City` | 3 unique cities: Madrid, Sevilla, Barcelona | Label Encoding + OHE + Dummy |
| Scale mismatch | `Age` vs `Income` | Age: 25–45 / Income: 30,000–50,000 — 1,000x difference | Min-Max + Z-Score |

**Scale mismatch business risk:** A distance-based model (KNN, SVM, KMeans) would assign 99.9% of its weight to `Income` and effectively ignore `Age`, distorting customer segmentation.

---

## 3. Phase 3 — Data Preparation <a id='phase-3'></a>

### 3.1 Missing Value Imputation — Mean Strategy <a id='imputation'></a>

In [ ]:
# =============================================================================
# STEP 1 — MEAN IMPUTATION FOR INCOME
# Strategy: SimpleImputer with strategy='mean'
# Rationale: small dataset (n=4), no extreme outliers in known values
# =============================================================================

from sklearn.impute import SimpleImputer

df_prep = df.copy()

imputer = SimpleImputer(strategy='mean')
df_prep['Income'] = imputer.fit_transform(df_prep[['Income']])

income_mean = df['Income'].mean()

print('=== Mean Imputation Result ===')
print(f'Income mean (from known values): ${income_mean:,.0f}')
print()
print(df_prep[['ID', 'Age', 'City', 'Income']])
print()
print(f'Missing values remaining: {df_prep.isnull().sum().sum()}')

**Result:** Row 3 (ID=3) imputed with mean income = $40,000 — computed from rows 1, 2, and 4.

> **Business note:** Mean imputation is conservative — it does not introduce bias in the mean but slightly reduces variance. For this 4-row demo dataset it is the correct choice. In production with larger datasets, consider median imputation (if outliers present) or KNN imputation (if geographic/demographic correlation is available).

### 3.2 Categorical Encoding — Label Encoding <a id='label-encoding'></a>

In [ ]:
# =============================================================================
# STEP 2 — LABEL ENCODING FOR CITY
# Assigns an integer to each unique category (alphabetical order by default)
# Warning: implies ordinal relationship — use only for tree-based models
# =============================================================================

le = LabelEncoder()

df_prep['City_LabelEncoded'] = le.fit_transform(df_prep['City'])

print('=== Label Encoding Result ===')
print('Mapping:')
for city, code in zip(le.classes_, range(len(le.classes_))):
    print(f'  {city:<12} → {code}')
print()
print(df_prep[['ID', 'City', 'City_LabelEncoded']])

**Result:** Barcelona=0, Madrid=1, Sevilla=2 (alphabetical).

> **Business caution:** Label Encoding introduces a false ordinal relationship (Sevilla=2 > Madrid=1 > Barcelona=0). A linear regression or neural network would treat Sevilla as "twice as much" as Madrid, which is meaningless. Use Label Encoding only with tree-based models (Decision Tree, Random Forest, XGBoost) that split on thresholds — not with distance-based models.

### 3.3 Categorical Encoding — One-Hot Encoding <a id='ohe'></a>

In [ ]:
# =============================================================================
# STEP 3 — ONE-HOT ENCODING FOR CITY
# Creates one binary column per category — no ordinal relationship implied
# =============================================================================

from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
city_ohe = ohe.fit_transform(df_prep[['City']])

# Build column names from encoder categories
ohe_columns = [f'City_{cat}' for cat in ohe.categories_[0]]
df_ohe = pd.DataFrame(city_ohe, columns=ohe_columns, dtype=int)

df_prep_ohe = pd.concat([df_prep.reset_index(drop=True), df_ohe], axis=1)

print('=== One-Hot Encoding Result ===')
print(df_prep_ohe[['ID', 'City'] + ohe_columns])

**Result:** 3 binary columns — one per city. Each row has exactly one `1` and two `0`s.

> **Business note:** OHE is the standard choice for nominal categorical variables in linear models, SVM, and neural networks. Drawback: creates K columns for K categories (curse of dimensionality in high-cardinality features like postal codes or product SKUs).

### 3.4 Dummy Variables (Drop-First Encoding) <a id='dummy'></a>

In [ ]:
# =============================================================================
# STEP 4 — DUMMY VARIABLES WITH DROP_FIRST=TRUE
# Drops the first category to avoid multicollinearity (dummy variable trap)
# Dropped reference category: Barcelona (alphabetically first)
# =============================================================================

df_dummy = pd.get_dummies(
    df_prep[['ID', 'Age', 'City', 'Income']],
    columns=['City'],
    drop_first=True,
    dtype=int
)

print('=== Dummy Variables Result (drop_first=True) ===')
print(f'Reference category dropped: Barcelona')
print()
print(df_dummy)

**Result:** Only 2 columns (Madrid, Sevilla) instead of 3. Barcelona is implicitly encoded when both dummies = 0.

> **Why drop_first matters:** In regression models, including all K dummy columns creates perfect multicollinearity (the dummy trap) — the model cannot invert the design matrix. Dropping the reference category (Barcelona) solves this. Coefficients for Madrid and Sevilla are then interpreted as the income difference *relative to* Barcelona customers.

### 3.5 Feature Scaling — Min-Max Normalization <a id='minmax'></a>

In [ ]:
# =============================================================================
# STEP 5 — MIN-MAX NORMALIZATION
# Formula: x_norm = (x - x_min) / (x_max - x_min)
# Result: all values scaled to [0, 1]
# =============================================================================

# Use the imputed dataset (ID excluded — not a feature)
df_scaling = df_prep[['Age', 'Income']].copy()

scaler_minmax = MinMaxScaler()
scaled_minmax = scaler_minmax.fit_transform(df_scaling)

df_minmax = pd.DataFrame(
    scaled_minmax,
    columns=['Age_MinMax', 'Income_MinMax']
)

print('=== Min-Max Normalization Result ===')
print(f'Age range:    [{df_scaling["Age"].min()}, {df_scaling["Age"].max()}]  →  [0.0000, 1.0000]')
print(f'Income range: [{df_scaling["Income"].min():,.0f}, {df_scaling["Income"].max():,.0f}]  →  [0.0000, 1.0000]')
print()
print(pd.concat([df_prep[['ID']].reset_index(drop=True), df_minmax], axis=1))

### 3.6 Feature Scaling — Z-Score Standardization <a id='zscore'></a>

In [ ]:
# =============================================================================
# STEP 6 — Z-SCORE STANDARDIZATION
# Formula: x_std = (x - mean) / std_dev
# Result: mean=0, std=1 (approximately, with ddof correction for small n)
# =============================================================================

scaler_zscore = StandardScaler()
scaled_zscore = scaler_zscore.fit_transform(df_scaling)

df_zscore = pd.DataFrame(
    scaled_zscore,
    columns=['Age_ZScore', 'Income_ZScore']
)

print('=== Z-Score Standardization Result ===')
print(f'Age    — Mean: {df_scaling["Age"].mean():.1f},   Std: {df_scaling["Age"].std():.2f}')
print(f'Income — Mean: {df_scaling["Income"].mean():,.1f}, Std: {df_scaling["Income"].std():,.2f}')
print()
print(pd.concat([df_prep[['ID']].reset_index(drop=True), df_zscore], axis=1))
print()
print('Post-scaling verification:')
print(f'Age    — Mean: {df_zscore["Age_ZScore"].mean():.4f},  Std: {df_zscore["Age_ZScore"].std():.4f}')
print(f'Income — Mean: {df_zscore["Income_ZScore"].mean():.4f},  Std: {df_zscore["Income_ZScore"].std():.4f}')

### 3.7 Scaling Comparison Visualization <a id='visualization'></a>

In [ ]:
# =============================================================================
# VISUALIZATION — Raw vs Min-Max vs Z-Score comparison
# =============================================================================

customer_labels = ['C1\n(Madrid)', 'C2\n(Sevilla)', 'C3\n(Madrid)', 'C4\n(Barcelona)']
x = np.arange(len(customer_labels))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    'Feature Scaling Comparison — Age & Income\nRetailData Analytics — L3 Case Study',
    fontsize=13, fontweight='bold', y=1.02
)

# --- Plot 1: Raw values ---
ax1 = axes[0]
ax1.bar(x - width/2, df_prep['Age'],    width, label='Age',    color='steelblue', alpha=0.85)
ax1.bar(x + width/2, df_prep['Income'], width, label='Income', color='coral',     alpha=0.85)
ax1.set_title('Raw Values\n(before scaling)', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels(customer_labels)
ax1.set_ylabel('Value')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.annotate('Scale mismatch:\nAge 25–45 vs Income 30k–50k',
             xy=(0.5, 0.92), xycoords='axes fraction',
             ha='center', fontsize=8, color='darkred')

# --- Plot 2: Min-Max ---
ax2 = axes[1]
ax2.bar(x - width/2, df_minmax['Age_MinMax'],    width, label='Age',    color='steelblue', alpha=0.85)
ax2.bar(x + width/2, df_minmax['Income_MinMax'], width, label='Income', color='coral',     alpha=0.85)
ax2.set_title('Min-Max Normalization\n(range [0, 1])', fontsize=11)
ax2.set_xticks(x)
ax2.set_xticklabels(customer_labels)
ax2.set_ylabel('Normalized Value')
ax2.set_ylim(-0.1, 1.2)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.axhline(y=0, color='gray', linewidth=0.8, linestyle='--')
ax2.axhline(y=1, color='gray', linewidth=0.8, linestyle='--')

# --- Plot 3: Z-Score ---
ax3 = axes[2]
ax3.bar(x - width/2, df_zscore['Age_ZScore'],    width, label='Age',    color='steelblue', alpha=0.85)
ax3.bar(x + width/2, df_zscore['Income_ZScore'], width, label='Income', color='coral',     alpha=0.85)
ax3.set_title('Z-Score Standardization\n(mean=0, std=1)', fontsize=11)
ax3.set_xticks(x)
ax3.set_xticklabels(customer_labels)
ax3.set_ylabel('Standardized Value (σ)')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)
ax3.axhline(y=0, color='gray', linewidth=1.2, linestyle='--', label='Mean (0)')

plt.tight_layout()
plt.savefig('../reports/scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to reports/scaling_comparison.png')

---

## 4. Phase 5 — Evaluation & Reflection <a id='phase-5'></a>

### Preprocessing Summary Table

| Step | Technique | Input | Output | Business Rationale |
|---|---|---|---|---|
| 1 | Mean Imputation | `Income` with 1 NaN | 0 missing values | Conservative fill — preserves mean without introducing bias toward extreme values |
| 2 | Label Encoding | `City` (3 categories) | `City_LabelEncoded` (0–2) | Compact single-column representation; safe for tree-based models only |
| 3 | One-Hot Encoding | `City` (3 categories) | 3 binary columns | Removes ordinal assumption; preferred for linear and distance-based models |
| 4 | Dummy Variables | `City` (3 categories) | 2 binary columns | Avoids dummy trap in regression models; Barcelona = reference category |
| 5 | Min-Max Normalization | `Age`, `Income` | Values in [0, 1] | Equalizes scale when feature bounds are known; preserves proportional distances |
| 6 | Z-Score Standardization | `Age`, `Income` | Mean=0, Std=1 | Centers distribution; robust to outliers in production data; preferred for Logistic Regression, SVM, Neural Networks |

### Reflection Q&A

**Q1: Why is preprocessing essential before training an ML model?**

Machine learning algorithms operate on mathematical operations — matrix multiplications, distance calculations, gradient updates. They cannot interpret text categories, cannot handle missing values (most implementations will throw errors or silently impute with 0), and assign implicit importance to features with larger numeric ranges. A model trained on unprocessed data will learn noise and scale artifacts rather than true patterns. Preprocessing is not a bureaucratic step — it is the direct translation of business meaning into a mathematical representation the model can learn from. In LEAN terms: **garbage in, garbage out is the highest-cost defect in any data pipeline**.

**Q2: What are the key differences between normalization and standardization?**

| Dimension | Min-Max Normalization | Z-Score Standardization |
|---|---|---|
| **Output range** | Bounded [0, 1] | Unbounded (centered at 0) |
| **Sensitive to outliers** | Yes — one extreme value compresses all others | Less sensitive — uses mean and std |
| **Assumes distribution** | No assumption | Implicitly assumes approximately normal |
| **Use when** | Known, fixed bounds; neural networks with sigmoid/tanh | Unknown bounds; linear models, SVM, PCA, neural nets with ReLU |
| **Production risk** | New data outside training range → values outside [0,1] | New data handled gracefully as z-score |

**Personal perspective:** For the RetailData use case — predicting recurring customers — Z-Score is the safer production choice because new customer data arriving after model deployment may have income values outside the training range (e.g., a high-income customer in a new market). Min-Max would produce values > 1 for that customer, potentially destabilizing the model. Z-Score simply produces a larger positive z-value and the model handles it correctly.

In [ ]:
# =============================================================================
# FINAL PREPROCESSED DATASET — Export
# Includes: imputed Income, Label Encoding, OHE columns, Dummy columns,
# Min-Max scaled, Z-Score scaled
# =============================================================================

from pathlib import Path

df_final = pd.concat([
    df_prep[['ID', 'Age', 'City', 'Income', 'City_LabelEncoded']].reset_index(drop=True),
    df_ohe.reset_index(drop=True),
    df_minmax.reset_index(drop=True),
    df_zscore.reset_index(drop=True)
], axis=1)

# Create output directory if it does not exist
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'retaildata_preprocessed.csv'
df_final.to_csv(output_path, index=False)

print('=== Final Preprocessed Dataset ===')
print(df_final.to_string())
print()
print(f'File exported successfully.' if output_path.exists() else 'Export failed.')
print(f'Shape: {df_final.shape}')

---

## 5. LEAN Filter — Waste Elimination Review <a id='lean-filter'></a>

| LEAN Waste Type | Identified? | Detail | Eliminated? |
|---|---|---|---|
| **Defects** | ✅ Yes | 1 missing value in `Income` (25% of column) | ✅ Mean imputation applied |
| **Overprocessing** | ✅ Yes | Three encoding methods demonstrated (Label + OHE + Dummy) for the same column — in production, only one would be selected | ⚠️ Intentional for learning; in production use only one |
| **Waiting** | ✅ Yes | Unscaled features cause model convergence delays in gradient-based algorithms | ✅ Scaling applied |
| **Inventory (data)** | ✅ Yes | Raw dataset stored separately; only processed version forwarded to modeling | ✅ `data/raw/` preserved; `data/processed/` for pipeline |
| **Motion** | ❌ No | Single notebook, linear execution — no unnecessary navigation | N/A |
| **Transport** | ❌ No | No unnecessary data movement between systems | N/A |

---

## 6. Decisions Log <a id='decisions-log'></a>

| # | Decision | Rationale | LEAN Value | Alternative Considered |
|---|---|---|---|---|
| D1 | Mean imputation for `Income` | n=3 known values, no outliers, symmetric distribution | Eliminates defect (missing data) | Median (same result here); KNN imputation (overkill for 4 rows) |
| D2 | Label Encoding demonstrated but flagged | Bootcamp requirement; explicitly documented ordinal-relationship risk | Transparency — prevents downstream modeling errors | Skip entirely (but required by case) |
| D3 | One-Hot Encoding as preferred nominal strategy | No ordinal assumption; compatible with linear and distance-based models | Prevents silent model bias | Target Encoding (requires target variable not available here) |
| D4 | drop_first=True in Dummy encoding | Avoids multicollinearity (dummy trap) in regression models | Eliminates structural defect in model matrix | drop_first=False (incorrect for regression) |
| D5 | Both scalers applied and compared | Case requirement; comparison provides model-selection decision framework | Knowledge reuse across future projects | Apply only Z-Score (recommended for production) |
| D6 | Z-Score recommended for production | Handles out-of-range new data gracefully; preferred for Logistic Regression and SVM | Reduces production risk | Min-Max (fragile to distribution shift) |

---

## 7. Next Steps <a id='next-steps'></a>

| Priority | Next Step | Module |
|---|---|---|
| 🔴 Immediate | Apply this preprocessing pipeline to the full RetailData customer dataset (when available) using `Pipeline` + `ColumnTransformer` from scikit-learn | M5 |
| 🟡 Short-term | Train a Logistic Regression classifier on the preprocessed features to predict recurring purchase probability | M5–M6 |
| 🟢 Long-term | Integrate preprocessing as a reusable `src/preprocessing.py` module — paralleling the `wrangling.py` approach from the fintech wrangling case | Portfolio |

---

*Framework: CRISP-DM + LEAN | Methodology: Case-Based Learning (CBL)*

**Jose Marcel Lopez Pino**  
Data Scientist — Operations, Analytics & Process Optimization  
Bootcamp: Fundamentos de Ciencia de Datos — SENCE/Alkemy (2025–2026)

[![GitHub](https://img.shields.io/badge/GitHub-joselopezp-181717?style=flat&logo=github)](https://github.com/joselopezp)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-jose--lopez--pino-0077B5?style=flat&logo=linkedin)](https://www.linkedin.com/in/jose-lopez-pino/)